# 📱 Mobile Phone Price Prediction
### Exploratory Data Analysis & Model Comparison

**Goal:** Predict the price range category (0–3) of a mobile phone from its specifications.

| Label | Price Range |
|---|---|
| 0 | Budget — < ₹10,000 |
| 1 | Mid-Range — ₹10,000–₹20,000 |
| 2 | Upper Mid — ₹20,000–₹40,000 |
| 3 | Premium — > ₹40,000 |

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score

from data_preprocessing import generate_dataset, load_and_clean, encode_and_scale, get_train_test_split
from model_training      import define_models

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('✅ Setup complete')

---
## 1. Load / Generate Dataset

In [ ]:
DATA_PATH = '../data/mobile_data.csv'

if not os.path.exists(DATA_PATH):
    df_raw = generate_dataset(1000)
    df_raw.to_csv(DATA_PATH, index=False)
    print('Dataset generated and saved.')
else:
    print('Dataset loaded from disk.')

df = load_and_clean(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

---
## 2. Exploratory Data Analysis

In [ ]:
# Basic stats
df.describe()

In [ ]:
# Class distribution
price_counts = df['price_range'].value_counts().sort_index()
labels = ['Budget\n(<₹10k)', 'Mid\n(₹10-20k)', 'Upper Mid\n(₹20-40k)', 'Premium\n(>₹40k)']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, price_counts.values,
              color=['#2ecc71','#f39c12','#e67e22','#e74c3c'], edgecolor='white')
ax.bar_label(bars, padding=4)
ax.set_title('Price Range Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions
num_cols = ['ram_mb','storage_gb','battery_mah','screen_size_in',
            'camera_mp','processor_ghz','weight_g','resolution_ppi']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')
plt.suptitle('Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[num_cols + ['price_range']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# RAM vs Price Range (boxplot)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x='price_range', y='ram_mb', palette='Set2', ax=axes[0])
axes[0].set_xticklabels(labels)
axes[0].set_title('RAM vs Price Range')

sns.boxplot(data=df, x='price_range', y='camera_mp', palette='Set3', ax=axes[1])
axes[1].set_xticklabels(labels)
axes[1].set_title('Camera MP vs Price Range')

plt.tight_layout()
plt.show()

In [ ]:
# Brand distribution by price tier
brand_price = df.groupby(['brand','price_range']).size().unstack(fill_value=0)
brand_price.plot(kind='bar', stacked=True, figsize=(12, 5),
                 color=['#2ecc71','#f39c12','#e67e22','#e74c3c'])
plt.title('Brand Distribution by Price Range', fontsize=13, fontweight='bold')
plt.xlabel('Brand')
plt.ylabel('Count')
plt.legend(labels, title='Price Range')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 5G adoption by price tier
five_g_rate = df.groupby('price_range')['five_g'].mean() * 100
fig, ax = plt.subplots(figsize=(7, 4))
five_g_rate.plot(kind='bar', ax=ax,
                 color=['#2ecc71','#f39c12','#e67e22','#e74c3c'], edgecolor='white')
ax.set_xticklabels(labels, rotation=0)
ax.set_ylabel('5G Adoption (%)')
ax.set_title('5G Adoption by Price Tier', fontsize=12, fontweight='bold')
ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=3)
plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

X, y, feature_cols, le, scaler = encode_and_scale(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Features  : {feature_cols}')
print(f'Train size: {X_train.shape}')
print(f'Test size : {X_test.shape}')

---
## 4. Model Training & Comparison

In [ ]:
from sklearn.metrics import accuracy_score

models  = define_models()
results = {}

print(f'{"Model":<25} {"CV Mean":>9} {"CV Std":>8} {"Test Acc":>10}')
print('-' * 56)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    
    results[name] = {
        'model':     model,
        'cv_mean':   cv_scores.mean(),
        'cv_std':    cv_scores.std(),
        'test_acc':  test_acc,
        'train_acc': accuracy_score(y_train, model.predict(X_train)),
    }
    print(f'{name:<25} {cv_scores.mean():>9.4f} {cv_scores.std():>8.4f} {test_acc:>10.4f}')

In [ ]:
# Accuracy comparison chart
names      = list(results.keys())
train_accs = [results[n]['train_acc'] for n in names]
test_accs  = [results[n]['test_acc']  for n in names]
cv_means   = [results[n]['cv_mean']   for n in names]

x = np.arange(len(names))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w, train_accs, w, label='Train Accuracy', color='#4C72B0')
b2 = ax.bar(x,     cv_means,   w, label='CV Accuracy',    color='#55A868')
b3 = ax.bar(x + w, test_accs,  w, label='Test Accuracy',  color='#DD8452')

for bars in [b1, b2, b3]:
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Accuracy')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Best Model — Detailed Evaluation

In [ ]:
best_name  = max(results, key=lambda k: results[k]['test_acc'])
best_model = results[best_name]['model']
y_pred     = best_model.predict(X_test)

print(f'🏆 Best Model: {best_name}  (Test Accuracy = {results[best_name]["test_acc"]:.4f})')
print()
print(classification_report(y_test, y_pred,
                             target_names=['Budget','Mid-Range','Upper Mid','Premium']))

In [ ]:
# Confusion matrix
cm   = confusion_matrix(y_test, y_pred, labels=[0,1,2,3])
disp = ConfusionMatrixDisplay(cm, display_labels=['Budget','Mid','Upper Mid','Premium'])

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances (tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    sorted_idx  = np.argsort(importances)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh([feature_cols[i] for i in sorted_idx],
            importances[sorted_idx], color='#55A868')
    ax.set_xlabel('Importance Score')
    ax.set_title(f'Feature Importances — {best_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importances not available for this model.')

---
## 6. Save Best Model

In [ ]:
import joblib, os

os.makedirs('../models', exist_ok=True)
bundle = {
    'model':         best_model,
    'model_name':    best_name,
    'scaler':        scaler,
    'label_encoder': le,
    'feature_cols':  feature_cols,
}
joblib.dump(bundle, '../models/best_model.pkl')
print(f'✅ Best model ({best_name}) saved to ../models/best_model.pkl')

---
## 7. Quick Prediction Demo

In [ ]:
from predict import predict_phone, PRICE_LABELS

phones = [
    {'brand':'Realme',  'ram_mb':3072, 'storage_gb':64,  'battery_mah':4000,
     'screen_size_in':6.5, 'camera_mp':13,  'processor_ghz':1.8,
     'five_g':0, 'weight_g':185, 'resolution_ppi':270},
    
    {'brand':'OnePlus', 'ram_mb':8192, 'storage_gb':128, 'battery_mah':4500,
     'screen_size_in':6.7, 'camera_mp':50,  'processor_ghz':3.0,
     'five_g':1, 'weight_g':190, 'resolution_ppi':450},
    
    {'brand':'Apple',   'ram_mb':8192, 'storage_gb':256, 'battery_mah':3877,
     'screen_size_in':6.1, 'camera_mp':48,  'processor_ghz':3.5,
     'five_g':1, 'weight_g':174, 'resolution_ppi':460},
]

loaded_bundle = joblib.load('../models/best_model.pkl')
print(f"{'Brand':<10} {'Prediction':<5} {'Label'}")
print('-' * 55)
for p in phones:
    pred, label = predict_phone(p, loaded_bundle)
    print(f"{p['brand']:<10} {pred:<5} {label}")